## Sparse, Contractive, and Variational Auto-Encoders


# Setup

In [ ]:
import argparse
import os
import sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import MNIST
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## Set random seeds for reproducibility

In [ ]:
def set_seed(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(123)

## Device configuration (Optional)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


# Load data


### MNIST dataset

In [ ]:
def load_mnist_data(data_dir, batch_size):
    # Load MNIST dataset with proper normalization
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))  # Normalize to [-1, 1]
    ])

    train_dataset = MNIST(root=data_dir, train=True, download=True, transform=transform)
    test_dataset = MNIST(root=data_dir, train=False, download=True, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                             num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                            num_workers=4, pin_memory=True)

    print(f"MNIST loaded: {len(train_dataset)} training samples, {len(test_dataset)} test samples")
    return train_loader, test_loader

### Frey Face dataset

In [ ]:
# Custom class dataset for Frey Face data
class FreyFaceDataset(Dataset):
    def __init__(self, data_dir, train=True):
        self.data_dir = data_dir
        self.train = train

        # Download and load Frey Face data
        self.data = self._load_frey_data()

        # Split into train/validation (90%/10%)
        n_samples = len(self.data)
        indices = np.arange(n_samples)
        np.random.shuffle(indices)

        split_idx = int(0.9 * n_samples)
        if train:
            self.data = self.data[indices[:split_idx]]
        else:
            self.data = self.data[indices[split_idx:]]

    # Load Frey Face dataset from file or download
    def _load_frey_data(self):
        frey_file = os.path.join(self.data_dir, 'frey_rawface.npy')

        if os.path.exists(frey_file):
            data = np.load(frey_file)
        else:
            # Try to download and process Frey Face data
            print("Downloading Frey Face dataset...")
            try:
                from scipy.io import loadmat
                import urllib.request

                # Try alternative URL
                url = 'https://cs.nyu.edu/~roweis/data/frey_rawface.mat'
                mat_file = os.path.join(self.data_dir, 'frey_rawface.mat')
                urllib.request.urlretrieve(url, mat_file)

                # Load and process
                mat_data = loadmat(mat_file)
                data = mat_data['ff'].T  # Shape: (1965, 560)
                data = data.reshape(-1, 28, 20)  # Reshape to images

                # Save for future use
                np.save(frey_file, data)
                print("Successfully downloaded Frey Face dataset")

            except Exception as e:
                print(f"Error downloading Frey Face data: {e}")
                print("Creating synthetic face-like data for demonstration...")
                # Create more realistic synthetic data
                np.random.seed(42)  # For reproducibility
                data = np.zeros((1965, 28, 20))

                # Create synthetic face-like patterns
                for i in range(1965):
                    # Create a basic face-like pattern
                    face = np.random.rand(28, 20) * 0.3 + 0.3  # Base face

                    # Add eyes
                    face[8:12, 6:8] = np.random.rand(4, 2) * 0.4 + 0.1
                    face[8:12, 12:14] = np.random.rand(4, 2) * 0.4 + 0.1

                    # Add nose
                    face[12:16, 9:11] = np.random.rand(4, 2) * 0.3 + 0.4

                    # Add mouth
                    face[18:20, 8:12] = np.random.rand(2, 4) * 0.3 + 0.2

                    # Add some variation
                    face += np.random.rand(28, 20) * 0.1

                    data[i] = face

                # Save synthetic data
                np.save(frey_file, data)
                print("Synthetic face dataset created and saved")

        # Normalize to [0, 1] and then to [-1, 1]
        if data.max() > 1.0:
            data = data.astype(np.float32) / 255.0
        else:
            data = data.astype(np.float32)
        data = (data - 0.5) / 0.5

        return data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = self.data[idx]
        # Convert to tensor and add channel dimension
        image = torch.from_numpy(image).unsqueeze(0)  # Shape: (1, 28, 20)
        return image, 0  # Return dummy label for consistency

# Load Frey Face dataset
def load_frey_data(data_dir, batch_size):
    train_dataset = FreyFaceDataset(data_dir, train=True)
    val_dataset = FreyFaceDataset(data_dir, train=False)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                             num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                           num_workers=4, pin_memory=True)

    print(f"Frey Face loaded: {len(train_dataset)} training samples, {len(val_dataset)} validation samples")
    return train_loader


# Utility functions

In [ ]:
# Utility functions

# Calculate PSNR between two images
def psnr(img1, img2):
    mse = F.mse_loss(img1, img2, reduction='mean')
    if mse == 0:
        return float('inf')
    return 10 * torch.log10(1.0 / mse)

# Save a grid of images
def save_image_grid(images, filename, nrow=10, normalize=True):
    from torchvision.utils import save_image
    save_image(images, filename, nrow=nrow, normalize=normalize, value_range=(-1, 1))

# AutoEncoder

## Unet AutoEncoder without skip connections

In [ ]:
# Network Architectures

# U-Net based AutoEncoder without skip connections
class UNetAutoEncoder(nn.Module):

    def __init__(self, in_channels=1, channels=[32, 64, 128, 256], latent_dim=256):
        super(UNetAutoEncoder, self).__init__()
        self.channels = channels
        self.latent_dim = latent_dim

        # Encoder - 4 layers to get from 28x28 to 1x1 (approximately)
        self.encoder = nn.Sequential(
            # First block: 28x28 -> 14x14
            nn.Conv2d(in_channels, channels[0], 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels[0], channels[0], 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Second block: 14x14 -> 7x7
            nn.Conv2d(channels[0], channels[1], 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels[1], channels[1], 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Third block: 7x7 -> 3x3 (roughly)
            nn.Conv2d(channels[1], channels[2], 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels[2], channels[2], 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Fourth block: 3x3 -> 1x1 (roughly)
            nn.Conv2d(channels[2], channels[3], 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels[3], channels[3], 3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1) # Use adaptive pooling to get 1x1 regardless of exact size
        )

        # Calculate flattened size after encoder
        self.flatten_size = channels[3] # Size is the number of channels after adaptive pooling

        # Latent space
        self.fc_encode = nn.Linear(self.flatten_size, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, self.flatten_size)

        # Decoder - reverse of encoder with exact size specifications
        self.decoder = nn.Sequential(
            # Start from 1x1 and upsample
            nn.Upsample(size=(3, 3), mode='bilinear', align_corners=True),
            nn.Conv2d(channels[3], channels[2], 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels[2], channels[2], 3, padding=1),
            nn.ReLU(inplace=True),

            # First upsample: 3x3 -> 7x7
            nn.Upsample(size=(7, 7), mode='bilinear', align_corners=True),
            nn.Conv2d(channels[2], channels[1], 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels[1], channels[1], 3, padding=1),
            nn.ReLU(inplace=True),

            # Second upsample: 7x7 -> 14x14
            nn.Upsample(size=(14, 14), mode='bilinear', align_corners=True),
            nn.Conv2d(channels[1], channels[0], 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels[0], channels[0], 3, padding=1),
            nn.ReLU(inplace=True),

            # Third upsample: 14x14 -> 28x28
            nn.Upsample(size=(28, 28), mode='bilinear', align_corners=True),
            nn.Conv2d(channels[0], in_channels, 3, padding=1),
            nn.Tanh()
        )


    def encode(self, x):
        """Encode input to latent space"""
        x = self.encoder(x)
        x = x.view(x.size(0), -1)  # Flatten
        h = self.fc_encode(x)
        return h

    def decode(self, h):
        """Decode latent representation to image"""
        x = self.fc_decode(h)
        # Reshape to the size before the adaptive pooling in the encoder
        x = x.view(x.size(0), self.channels[3], 1, 1)
        x = self.decoder(x)
        return x

    def forward(self, x):
        h = self.encode(x)
        x_recon = self.decode(h)
        return x_recon, h

## Sparse AutoEncoder with KL sparsity penalty

In [ ]:
class SparseAutoEncoder(UNetAutoEncoder):

    def __init__(self, in_channels=1, channels=[32, 64, 128, 256], latent_dim=256,
                 rho=0.05, beta=1.0):
        super(SparseAutoEncoder, self).__init__(in_channels, channels, latent_dim)
        self.rho = rho  # Target sparsity
        self.beta = beta  # Sparsity penalty weight

    def forward(self, x):
        h = self.encode(x)
        x_recon = self.decode(h)

        # Calculate sparsity penalty
        rho_hat = torch.mean(torch.sigmoid(h), dim=0)  # Average activation
        sparsity_penalty = self.beta * torch.sum(
            self.rho * torch.log(self.rho / rho_hat) +
            (1 - self.rho) * torch.log((1 - self.rho) / (1 - rho_hat))
        )

        return x_recon, h, sparsity_penalty

## Contractive AutoEncoder with Jacobian penalty

In [ ]:
class ContractiveAutoEncoder(UNetAutoEncoder):

    def __init__(self, in_channels=1, channels=[32, 64, 128, 256], latent_dim=256,
                 lambda_reg=1e-4):
        super(ContractiveAutoEncoder, self).__init__(in_channels, channels, latent_dim)
        self.lambda_reg = lambda_reg

    def forward(self, x):
        # Enable gradient computation for input
        x.requires_grad_(True)

        h = self.encode(x)
        x_recon = self.decode(h)

        # Calculate Jacobian penalty
        if self.training:
            # Compute Jacobian norm
            h_sum = torch.sum(h, dim=1)
            jacobian = torch.autograd.grad(
                outputs=h_sum, inputs=x,
                grad_outputs=torch.ones_like(h_sum),
                create_graph=True, retain_graph=True
            )[0]

            contractive_penalty = self.lambda_reg * torch.sum(jacobian ** 2)
        else:
            contractive_penalty = torch.tensor(0.0, device=x.device)

        return x_recon, h, contractive_penalty

## Variational AutoEncoder for Frey Face dataset

In [ ]:
class VariationalAutoEncoder(nn.Module):

    def __init__(self, in_channels=1, latent_dim=20):
        super(VariationalAutoEncoder, self).__init__()
        self.latent_dim = latent_dim

        # Encoder for 28x20 images - simplified approach
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, 2, 1),  # 28x20 -> 14x10
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, 2, 1),          # 14x10 -> 7x5
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, 2, 1),         # 7x5 -> 4x3 (rounded up)
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),              # 4x3 -> 1x1
            nn.Flatten()
        )

        # Calculate flattened size
        self.flatten_size = 128  # After adaptive pooling

        # Latent space
        self.fc_mu = nn.Linear(self.flatten_size, latent_dim)
        self.fc_logvar = nn.Linear(self.flatten_size, latent_dim)

        # Decoder
        self.fc_decode = nn.Linear(latent_dim, self.flatten_size)

        self.decoder = nn.Sequential(
            nn.Upsample(size=(7, 5), mode='bilinear', align_corners=True),
            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(size=(14, 10), mode='bilinear', align_corners=True),
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(size=(28, 20), mode='bilinear', align_corners=True),
            nn.Conv2d(32, in_channels, 3, padding=1),
            nn.Tanh()
        )

    def encode(self, x):
        """Encode input to latent parameters"""
        x = self.encoder(x)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        """Reparameterization trick"""
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        else:
            return mu

    def decode(self, z):
        """Decode latent representation to image"""
        x = self.fc_decode(z)
        x = x.view(x.size(0), 128, 1, 1)
        x = self.decoder(x)
        return x

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar

# Training Functions

## Train Sparse AutoEncoder

In [ ]:
def train_sparse_ae(train_loader, test_loader, args):
    print("Initializing Sparse AutoEncoder...")
    model = SparseAutoEncoder(in_channels=1, latent_dim=256, rho=args.sae_rho, beta=args.sae_beta).to(device)
    optimizer = optim.Adam(model.parameters(), lr=args.lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    best_loss = float('inf')
    patience_counter = 0

    train_losses = []
    val_losses = []

    for epoch in range(args.epochs):
        # Training
        model.train()
        train_loss = 0.0
        train_recon_loss = 0.0
        train_sparse_loss = 0.0

        for batch_idx, (data, _) in enumerate(tqdm(train_loader, desc=f'Epoch {epoch+1}/{args.epochs}')):
            data = data.to(device)

            optimizer.zero_grad()
            x_recon, _, sparsity_penalty = model(data)

            # Loss computation
            recon_loss = F.mse_loss(x_recon, data)
            total_loss = recon_loss + sparsity_penalty

            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)  # Gradient clipping
            optimizer.step()

            train_loss += total_loss.item()
            train_recon_loss += recon_loss.item()
            train_sparse_loss += sparsity_penalty.item()

        # Validation
        model.eval()
        val_loss = 0.0
        val_recon_loss = 0.0
        val_sparse_loss = 0.0

        with torch.no_grad():
            for data, _ in test_loader:
                data = data.to(device)
                x_recon, _, sparsity_penalty = model(data)

                recon_loss = F.mse_loss(x_recon, data)
                total_loss = recon_loss + sparsity_penalty

                val_loss += total_loss.item()
                val_recon_loss += recon_loss.item()
                val_sparse_loss += sparsity_penalty.item()

        # Average losses
        train_loss /= len(train_loader)
        val_loss /= len(test_loader)
        train_recon_loss /= len(train_loader)
        val_recon_loss /= len(test_loader)
        train_sparse_loss /= len(train_loader)
        val_sparse_loss /= len(test_loader)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(f'Epoch {epoch+1}: Train Loss: {train_loss:.6f} (Recon: {train_recon_loss:.6f}, Sparse: {train_sparse_loss:.6f}), '
              f'Val Loss: {val_loss:.6f} (Recon: {val_recon_loss:.6f}, Sparse: {val_sparse_loss:.6f})')

        # Early stopping
        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(args.model_dir, 'sparse_ae_best.pth'))
        else:
            patience_counter += 1
            if patience_counter >= 3:
                print("Early stopping triggered")
                break

        scheduler.step(val_loss)

    # Save final model and training history
    torch.save(model.state_dict(), os.path.join(args.model_dir, 'sparse_ae_final.pth'))

    # Plot training curves
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Sparse AutoEncoder Training')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    # Save sample reconstructions
    model.eval()
    with torch.no_grad():
        sample_data, _ = next(iter(test_loader))
        sample_data = sample_data[:16].to(device)
        recon_data, _ = model(sample_data)[:2]

        # Create comparison grid
        comparison = torch.cat([sample_data[:8], recon_data[:8]], dim=0)
        save_image_grid(comparison, os.path.join(args.output_dir, 'sae_reconstruction.png'), nrow=8)

    plt.tight_layout()
    plt.savefig(os.path.join(args.output_dir, 'sae_training_curves.png'), dpi=300)
    plt.close()

    return model

## Train Contractive AutoEncoder

In [ ]:
def train_contractive_ae(train_loader, test_loader, args):

    print("Initializing Contractive AutoEncoder...")
    model = ContractiveAutoEncoder(in_channels=1, latent_dim=256, lambda_reg=args.cae_lambda_reg).to(device)
    optimizer = optim.Adam(model.parameters(), lr=args.lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    best_loss = float('inf')
    patience_counter = 0

    train_losses = []
    val_losses = []

    for epoch in range(args.epochs):
        # Training
        model.train()
        train_loss = 0.0
        train_recon_loss = 0.0
        train_contract_loss = 0.0

        for batch_idx, (data, _) in enumerate(tqdm(train_loader, desc=f'Epoch {epoch+1}/{args.epochs}')):
            data = data.to(device)

            optimizer.zero_grad()
            x_recon, _, contractive_penalty = model(data)

            # Loss computation
            recon_loss = F.mse_loss(x_recon, data)
            total_loss = recon_loss + contractive_penalty

            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)  # Gradient clipping
            optimizer.step()

            train_loss += total_loss.item()
            train_recon_loss += recon_loss.item()
            train_contract_loss += contractive_penalty.item()

        # Validation
        model.eval()
        val_loss = 0.0
        val_recon_loss = 0.0
        val_contract_loss = 0.0

        with torch.no_grad():
            for data, _ in test_loader:
                data = data.to(device)
                x_recon, _, contractive_penalty = model(data)

                recon_loss = F.mse_loss(x_recon, data)
                total_loss = recon_loss + contractive_penalty

                val_loss += total_loss.item()
                val_recon_loss += recon_loss.item()
                val_contract_loss += contractive_penalty.item()

        # Average losses
        train_loss /= len(train_loader)
        val_loss /= len(test_loader)
        train_recon_loss /= len(train_loader)
        val_recon_loss /= len(test_loader)
        train_contract_loss /= len(train_loader)
        val_contract_loss /= len(test_loader)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(f'Epoch {epoch+1}: Train Loss: {train_loss:.6f} (Recon: {train_recon_loss:.6f}, Contract: {train_contract_loss:.6f}), '
              f'Val Loss: {val_loss:.6f} (Recon: {val_recon_loss:.6f}, Contract: {val_contract_loss:.6f})')

        # Early stopping
        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(args.model_dir, 'contractive_ae_best.pth'))
        else:
            patience_counter += 1
            if patience_counter >= 3:
                print("Early stopping triggered")
                break

        scheduler.step(val_loss)

    # Save final model and training history
    torch.save(model.state_dict(), os.path.join(args.model_dir, 'contractive_ae_final.pth'))

    # Plot training curves
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Contractive AutoEncoder Training')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    # Save sample reconstructions
    model.eval()
    with torch.no_grad():
        sample_data, _ = next(iter(test_loader))
        sample_data = sample_data[:16].to(device)
        recon_data, _ = model(sample_data)[:2]

        # Create comparison grid
        comparison = torch.cat([sample_data[:8], recon_data[:8]], dim=0)
        save_image_grid(comparison, os.path.join(args.output_dir, 'cae_reconstruction.png'), nrow=8)

    plt.tight_layout()
    plt.savefig(os.path.join(args.output_dir, 'cae_training_curves.png'), dpi=300)
    plt.close()

    return model

## Train Variational AutoEncoder

In [ ]:
def train_vae(train_loader, args):

    print("Initializing Variational AutoEncoder...")
    model = VariationalAutoEncoder(in_channels=1, latent_dim=20).to(device)
    optimizer = optim.Adam(model.parameters(), lr=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    # KL annealing schedule
    def kl_weight(epoch, max_epochs=args.epochs):
        return min(1.0, epoch / (max_epochs * 0.5))

    train_losses = []
    recon_losses = []
    kl_losses = []

    for epoch in range(args.epochs):  # Use same epochs as specified in args
        model.train()
        train_loss = 0.0
        train_recon_loss = 0.0
        train_kl_loss = 0.0

        beta = kl_weight(epoch)

        for batch_idx, (data, _) in enumerate(tqdm(train_loader, desc=f'Epoch {epoch+1}/{args.epochs}')):
            data = data.to(device)

            optimizer.zero_grad()
            x_recon, mu, logvar = model(data)

            # Loss computation
            recon_loss = F.mse_loss(x_recon, data, reduction='sum') / data.size(0)
            kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / data.size(0)

            total_loss = recon_loss + beta * kl_loss

            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            train_loss += total_loss.item()
            train_recon_loss += recon_loss.item()
            train_kl_loss += kl_loss.item()

        # Average losses
        train_loss /= len(train_loader)
        train_recon_loss /= len(train_loader)
        train_kl_loss /= len(train_loader)

        train_losses.append(train_loss)
        recon_losses.append(train_recon_loss)
        kl_losses.append(train_kl_loss)

        print(f'Epoch {epoch+1}: Train Loss: {train_loss:.6f} (Recon: {train_recon_loss:.6f}, KL: {train_kl_loss:.6f}, Beta: {beta:.3f})')

        # Save model periodically
        if (epoch + 1) % max(1, args.epochs // 3) == 0:
            torch.save(model.state_dict(), os.path.join(args.model_dir, f'vae_epoch_{epoch+1}.pth'))

        scheduler.step()

    # Save final model
    torch.save(model.state_dict(), os.path.join(args.model_dir, 'vae_final.pth'))

    # Plot training curves
    plt.figure(figsize=(15, 4))
    plt.subplot(1, 3, 1)
    plt.plot(train_losses, label='Total Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('VAE Total Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 3, 2)
    plt.plot(recon_losses, label='Reconstruction Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('VAE Reconstruction Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 3, 3)
    plt.plot(kl_losses, label='KL Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('VAE KL Loss')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(args.output_dir, 'vae_training_curves.png'), dpi=300)
    plt.close()

    # Save sample reconstructions
    model.eval()
    with torch.no_grad():
        sample_data, _ = next(iter(train_loader))
        sample_data = sample_data[:16].to(device)
        x_recon, _, _ = model(sample_data)

        # Create comparison grid
        comparison = torch.cat([sample_data[:8], x_recon[:8]], dim=0)
        save_image_grid(comparison, os.path.join(args.output_dir, 'vae_reconstruction.png'), nrow=8)

    return model

## Generate t-SNE plots for embeddings

In [ ]:
# Evaluation Functions

def generate_tsne_plots(model, test_loader, model_name, args):
    print(f"Generating t-SNE plots for {model_name}...")

    model.eval()
    embeddings = []
    labels = []

    with torch.no_grad():
        for data, target in tqdm(test_loader, desc="Extracting embeddings"):
            data = data.to(device)
            if hasattr(model, 'encode'):
                h = model.encode(data)
            else:
                _, h = model(data)
            embeddings.append(h.cpu().numpy())
            labels.append(target.numpy())

    # Concatenate all embeddings and labels
    embeddings = np.concatenate(embeddings, axis=0)
    labels = np.concatenate(labels, axis=0)

    # Standardize embeddings
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    embeddings_scaled = scaler.fit_transform(embeddings)

    # Apply t-SNE
    tsne = TSNE(n_components=2, perplexity=30, n_iter=2000, random_state=123)
    embeddings_2d = tsne.fit_transform(embeddings_scaled)

    # Create scatter plot
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1],
                         c=labels, cmap='tab10', alpha=0.6, s=1)
    plt.colorbar(scatter, label='Digit Class')
    plt.title(f't-SNE Visualization - {model_name}')
    plt.xlabel('t-SNE Component 1')
    plt.ylabel('t-SNE Component 2')
    plt.grid(True, alpha=0.3)

    # Save the plot
    plt.savefig(os.path.join(args.output_dir, 'tsne', f'{model_name.lower()}_tsne.png'),
                dpi=300, bbox_inches='tight')
    plt.close()

    print(f"t-SNE plot saved for {model_name}")
    return embeddings_2d

## Linear interpolation experiment

In [ ]:
def interpolation_experiment(model, test_loader, model_name, args):
    print(f"Running interpolation experiment for {model_name}...")

    model.eval()

    # Collect test data by class
    class_data = {i: [] for i in range(10)}
    with torch.no_grad():
        for data, target in test_loader:
            for i in range(len(target)):
                class_data[target[i].item()].append(data[i])

    # Convert to tensors
    for class_id in class_data:
        class_data[class_id] = torch.stack(class_data[class_id][:50])  # Take first 50 samples

    # Generate 20 pairs from different classes
    np.random.seed(123)
    pairs = []
    for _ in range(20):
        class1, class2 = np.random.choice(10, 2, replace=False)
        idx1 = np.random.randint(0, len(class_data[class1]))
        idx2 = np.random.randint(0, len(class_data[class2]))
        pairs.append((class_data[class1][idx1], class_data[class2][idx2]))

    # Interpolation parameters
    alphas = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

    psnr_values = []
    l2_distances = []

    for pair_idx, (img1, img2) in enumerate(pairs):
        img1, img2 = img1.to(device), img2.to(device)

        # Get embeddings
        with torch.no_grad():
            h1 = model.encode(img1.unsqueeze(0))
            h2 = model.encode(img2.unsqueeze(0))

        # Create interpolation grid
        interpolated_imgs = []
        true_reconstructions = []

        for alpha in alphas:
            # True interpolation in image space
            img_alpha = alpha * img1 + (1 - alpha) * img2

            # Approximate interpolation in latent space
            h_alpha_approx = alpha * h1 + (1 - alpha) * h2

            with torch.no_grad():
                # True reconstruction
                h_alpha_true = model.encode(img_alpha.unsqueeze(0))
                recon_true = model.decode(h_alpha_true)

                # Approximate reconstruction
                recon_approx = model.decode(h_alpha_approx)

                interpolated_imgs.append(recon_approx.squeeze(0))
                true_reconstructions.append(recon_true.squeeze(0))

                # Calculate PSNR and L2 distance
                psnr_val = psnr(recon_true, recon_approx)
                l2_dist = torch.norm(h_alpha_true - h_alpha_approx, p=2).item()

                # Handle case where psnr_val might be float or tensor
                if isinstance(psnr_val, torch.Tensor):
                    psnr_values.append(psnr_val.item())
                else:
                    psnr_values.append(psnr_val)
                l2_distances.append(l2_dist)

        # Create comparison grid for this pair
        true_row = torch.cat(true_reconstructions, dim=2)  # Concatenate along width
        approx_row = torch.cat(interpolated_imgs, dim=2)
        comparison = torch.cat([true_row, approx_row], dim=1)  # Stack vertically

        # Save individual pair
        save_image_grid(comparison.unsqueeze(0),
                       os.path.join(args.output_dir, 'interpolation', f'{model_name.lower()}_pair_{pair_idx}.png'),
                       nrow=1, normalize=True)

    # Save statistics
    stats_df = pd.DataFrame({
        'PSNR': psnr_values,
        'L2_Distance': l2_distances
    })
    stats_df.to_csv(os.path.join(args.output_dir, f'{model_name.lower()}_interpolation_stats.csv'), index=False)

    print(f"Interpolation experiment completed for {model_name}")
    print(f"Average PSNR: {np.mean(psnr_values):.4f}")
    print(f"Average L2 Distance: {np.mean(l2_distances):.4f}")

    return stats_df


## Train classifier on embeddings

In [ ]:
def train_classifier_on_embeddings(sae_model, cae_model, test_loader, args):
    print("Training classifiers on embeddings...")

    # Extract embeddings and labels
    sae_embeddings, cae_embeddings, labels = [], [], []

    sae_model.eval()
    cae_model.eval()

    with torch.no_grad():
        for data, target in tqdm(test_loader, desc="Extracting embeddings"):
            data = data.to(device)

            sae_h = sae_model.encode(data)
            cae_h = cae_model.encode(data)

            sae_embeddings.append(sae_h.cpu().numpy())
            cae_embeddings.append(cae_h.cpu().numpy())
            labels.append(target.numpy())

    # Concatenate all embeddings
    sae_embeddings = np.concatenate(sae_embeddings, axis=0)
    cae_embeddings = np.concatenate(cae_embeddings, axis=0)
    labels = np.concatenate(labels, axis=0)

    # Split into train/test
    from sklearn.model_selection import train_test_split

    # For SAE
    X_train_sae, X_test_sae, y_train, y_test = train_test_split(
        sae_embeddings, labels, test_size=0.3, random_state=123, stratify=labels
    )

    # For CAE
    X_train_cae, X_test_cae, _, _ = train_test_split(
        cae_embeddings, labels, test_size=0.3, random_state=123, stratify=labels
    )

    # Train classifiers
    results = {}

    # SVM Classifier
    print("Training SVM classifiers...")
    svm_sae = SVC(kernel='rbf', random_state=123)
    svm_cae = SVC(kernel='rbf', random_state=123)

    svm_sae.fit(X_train_sae, y_train)
    svm_cae.fit(X_train_cae, y_train)

    svm_sae_pred = svm_sae.predict(X_test_sae)
    svm_cae_pred = svm_cae.predict(X_test_cae)

    results['SVM_SAE'] = accuracy_score(y_test, svm_sae_pred)
    results['SVM_CAE'] = accuracy_score(y_test, svm_cae_pred)

    # Logistic Regression
    print("Training Logistic Regression classifiers...")
    lr_sae = LogisticRegression(random_state=123, max_iter=1000)
    lr_cae = LogisticRegression(random_state=123, max_iter=1000)

    lr_sae.fit(X_train_sae, y_train)
    lr_cae.fit(X_train_cae, y_train)

    lr_sae_pred = lr_sae.predict(X_test_sae)
    lr_cae_pred = lr_cae.predict(X_test_cae)

    results['LogReg_SAE'] = accuracy_score(y_test, lr_sae_pred)
    results['LogReg_CAE'] = accuracy_score(y_test, lr_cae_pred)

    # Create results table
    results_df = pd.DataFrame([
        ['Sparse AutoEncoder', results['SVM_SAE'], results['LogReg_SAE']],
        ['Contractive AutoEncoder', results['SVM_CAE'], results['LogReg_CAE']]
    ], columns=['Model', 'SVM Accuracy', 'Logistic Regression Accuracy'])

    print("\nClassifier Results:")
    print(results_df.to_string(index=False))

    # Save results
    results_df.to_csv(os.path.join(args.output_dir, 'classifier_results.csv'), index=False)

    return results_df

## Generate VAE latent traversals

In [ ]:
def generate_vae_latent_traversals(model, train_loader, args):
    print("Generating VAE latent traversals...")

    model.eval()

    # Get a sample to use as base
    sample_data, _ = next(iter(train_loader))
    sample_data = sample_data[:1].to(device)  # Take first sample

    with torch.no_grad():
        mu, logvar = model.encode(sample_data)
        base_z = model.reparameterize(mu, logvar)

    # Generate traversals for each latent dimension
    traversal_range = np.linspace(-3, 3, 10)

    for dim in range(20):  # 20 latent dimensions
        traversal_images = []

        for value in traversal_range:
            # Modify only the specific dimension
            z_modified = base_z.clone()
            z_modified[0, dim] = value

            with torch.no_grad():
                generated_img = model.decode(z_modified)
                traversal_images.append(generated_img.squeeze(0))

        # Create grid for this dimension
        traversal_grid = torch.stack(traversal_images, dim=0)
        save_image_grid(traversal_grid,
                       os.path.join(args.output_dir, 'traversals', f'latent_dim_{dim}.png'),
                       nrow=10, normalize=True)

    # Generate random samples
    print("Generating random samples...")
    with torch.no_grad():
        random_z = torch.randn(16, 20).to(device)
        random_samples = model.decode(random_z)
        save_image_grid(random_samples,
                       os.path.join(args.output_dir, 'vae_random_samples.png'),
                       nrow=4, normalize=True)

    print("VAE latent traversals completed")

## Evaluate all models and generate required outputs

In [ ]:
def evaluate_models(sae_model, cae_model, vae_model, test_loader, frey_loader, args):
    print("Starting comprehensive evaluation...")

    # Task 1a: t-SNE plots (20 marks)
    print("\n=== Task 1a: t-SNE Visualization ===")
    generate_tsne_plots(sae_model, test_loader, "Sparse_AutoEncoder", args)
    generate_tsne_plots(cae_model, test_loader, "Contractive_AutoEncoder", args)

    # Task 1b: Interpolation experiment (20 marks)
    print("\n=== Task 1b: Interpolation Experiment ===")
    sae_stats = interpolation_experiment(sae_model, test_loader, "Sparse_AutoEncoder", args)
    cae_stats = interpolation_experiment(cae_model, test_loader, "Contractive_AutoEncoder", args)

    # Task 1c: Classifier training (20 marks)
    print("\n=== Task 1c: Classifier Training ===")
    # Call the new function that includes SVC
    classifier_results = train_classifier_on_embeddings_with_svc(sae_model, cae_model, test_loader, args)

    # Task 2: VAE latent traversals (40 marks)
    print("\n=== Task 2: VAE Latent Traversals ===")
    generate_vae_latent_traversals(vae_model, frey_loader, args)

    print("\nAll evaluation tasks completed!")
    return {
        'sae_stats': sae_stats,
        'cae_stats': cae_stats,
        'classifier_results': classifier_results
    }

## Load and evaluate saved models

In [ ]:
def evaluate_saved_models(args):
    print("Loading saved models for evaluation...")

    # Load data
    _, test_loader = load_mnist_data(args.data_dir, args.batch_size)
    frey_loader = load_frey_data(args.data_dir, 128)

    # Load models
    sae_model = SparseAutoEncoder().to(device)
    cae_model = ContractiveAutoEncoder().to(device)
    vae_model = VariationalAutoEncoder().to(device)

    try:
        sae_model.load_state_dict(torch.load(os.path.join(args.model_dir, 'sparse_ae_best.pth')))
        cae_model.load_state_dict(torch.load(os.path.join(args.model_dir, 'contractive_ae_best.pth')))
        vae_model.load_state_dict(torch.load(os.path.join(args.model_dir, 'vae_final.pth')))

        # Evaluate
        evaluate_models(sae_model, cae_model, vae_model, test_loader, frey_loader, args)

    except FileNotFoundError as e:
        print(f"Model file not found: {e}")
        print("Please train the models first using --mode train_sae, train_cae, train_vae")

## Increase the accuracy by *hyperparameter tuning*

# Tune hyperparameters
  -  Tune(epochs and regularization strength) for the Sparse Autoencoder (SAE) and Contractive Autoencoder (CAE) training on the MNIST dataset.
  - Increase the training epochs to 30.

In [ ]:
# Modified function to include SVC classifier

def train_classifier_on_embeddings_with_svc(sae_model, cae_model, test_loader, args):
    print("Training classifiers on embeddings with SVC...")

    # Extract embeddings and labels
    sae_embeddings, cae_embeddings, labels = [], [], []

    sae_model.eval()
    cae_model.eval()

    with torch.no_grad():
        for data, target in tqdm(test_loader, desc="Extracting embeddings"):
            data = data.to(device)

            sae_h = sae_model.encode(data)
            cae_h = cae_model.encode(data)

            sae_embeddings.append(sae_h.cpu().numpy())
            cae_embeddings.append(cae_h.cpu().numpy())
            labels.append(target.numpy())

    # Concatenate all embeddings
    sae_embeddings = np.concatenate(sae_embeddings, axis=0)
    cae_embeddings = np.concatenate(cae_embeddings, axis=0)
    labels = np.concatenate(labels, axis=0)

    # Split into train/test
    from sklearn.model_selection import train_test_split

    # For SAE
    X_train_sae, X_test_sae, y_train, y_test = train_test_split(
        sae_embeddings, labels, test_size=0.3, random_state=123, stratify=labels
    )

    # For CAE
    X_train_cae, X_test_cae, _, _ = train_test_split(
        cae_embeddings, labels, test_size=0.3, random_state=123, stratify=labels
    )

    # Train classifiers
    results = {}

    # SVM Classifier
    print("Training SVM classifiers...")
    svm_sae = SVC(kernel='rbf', random_state=123)
    svm_cae = SVC(kernel='rbf', random_state=123)

    svm_sae.fit(X_train_sae, y_train)
    svm_cae.fit(X_train_cae, y_train)

    svm_sae_pred = svm_sae.predict(X_test_sae)
    svm_cae_pred = svm_cae.predict(X_test_cae)

    results['SVM_SAE'] = accuracy_score(y_test, svm_sae_pred)
    results['SVM_CAE'] = accuracy_score(y_test, svm_cae_pred)

    # Logistic Regression (keeping for comparison)
    print("Training Logistic Regression classifiers...")
    lr_sae = LogisticRegression(random_state=123, max_iter=1000)
    lr_cae = LogisticRegression(random_state=123, max_iter=1000)

    lr_sae.fit(X_train_sae, y_train)
    lr_cae.fit(X_train_cae, y_train)

    lr_sae_pred = lr_sae.predict(X_test_sae)
    lr_cae_pred = lr_cae.predict(X_test_cae)

    results['LogReg_SAE'] = accuracy_score(y_test, lr_sae_pred)
    results['LogReg_CAE'] = accuracy_score(y_test, lr_cae_pred)


    # Create results table
    results_df = pd.DataFrame([
        ['Sparse AutoEncoder', results['SVM_SAE'], results['LogReg_SAE']],
        ['Contractive AutoEncoder', results['SVM_CAE'], results['LogReg_CAE']]
    ], columns=['Model', 'SVM Accuracy', 'Logistic Regression Accuracy'])


    print("\nClassifier Results (with SVC and LogReg):")
    print(results_df.to_string(index=False))

    # Save results
    results_df.to_csv(os.path.join(args.output_dir, 'classifier_results_svc.csv'), index=False)

    return results_df

In [ ]:
import argparse
import os

def main():
    class Args:
        def __init__(self):
            # Set the desired mode, epochs, batch size, and regularization parameters here
            self.mode = 'all' # Change the mode here ('train_sae', 'evaluate', 'all')
            self.epochs = 30 # Increased epochs
            self.batch_size = 256
            self.lr = 1e-3
            self.data_dir = './data'
            self.output_dir = './outputs'
            self.model_dir = './models'
            self.sae_rho = 0.05  # Default sparse autoencoder rho
            self.sae_beta = 1e-3 # Experiment with sparse autoencoder beta
            self.cae_lambda_reg = 1e-5 # Experiment with contractive autoencoder lambda_reg

    args = Args()

    # Create directories
    os.makedirs(args.data_dir, exist_ok=True)
    os.makedirs(args.output_dir, exist_ok=True)
    os.makedirs(args.model_dir, exist_ok=True)
    os.makedirs(os.path.join(args.output_dir, 'interpolation'), exist_ok=True)
    os.makedirs(os.path.join(args.output_dir, 'tsne'), exist_ok=True)
    os.makedirs(os.path.join(args.output_dir, 'traversals'), exist_ok=True)


    print(f"Starting Deep Learning Assignment in mode: {args.mode}")

    if args.mode == 'all':
        print("Running complete pipeline...")
        # Load data
        train_loader, test_loader = load_mnist_data(args.data_dir, args.batch_size)
        frey_loader = load_frey_data(args.data_dir, 128)

        # Train all models
        print("Training Sparse AutoEncoder...")
        sae_model = train_sparse_ae(train_loader, test_loader, args)

        print("Training Contractive AutoEncoder...")
        cae_model = train_contractive_ae(train_loader, test_loader, args)

        print("Training Variational AutoEncoder...")
        vae_model = train_vae(frey_loader, args)

        # Evaluate all models
        print("Evaluating models...")
        evaluate_models(sae_model, cae_model, vae_model, test_loader, frey_loader, args)

    elif args.mode == 'train_sae':
        train_loader, test_loader = load_mnist_data(args.data_dir, args.batch_size)
        train_sparse_ae(train_loader, test_loader, args)

    elif args.mode == 'train_cae':
        train_loader, test_loader = load_mnist_data(args.data_dir, args.batch_size)
        train_contractive_ae(train_loader, test_loader, args)

    elif args.mode == 'train_vae':
        frey_loader = load_frey_data(args.data_dir, 128)
        train_vae(frey_loader, args)

    elif args.mode == 'evaluate':
        # Load trained models and evaluate
        evaluate_saved_models(args)

    print("Assignment completed successfully!")


if __name__ == '__main__':
    main()

Starting Deep Learning Assignment in mode: all
Running complete pipeline...
MNIST loaded: 60000 training samples, 10000 test samples
Frey Face loaded: 1768 training samples, 197 validation samples
Training Sparse AutoEncoder...
Initializing Sparse AutoEncoder...


Epoch 1/30: 100%|██████████| 235/235 [00:15<00:00, 15.31it/s]


Epoch 1: Train Loss: 0.460455 (Recon: 0.456260, Sparse: 0.004196), Val Loss: 0.458763 (Recon: 0.458762, Sparse: 0.000001)


Epoch 2/30: 100%|██████████| 235/235 [00:18<00:00, 12.61it/s]


Epoch 2: Train Loss: 0.448007 (Recon: 0.448006, Sparse: 0.000000), Val Loss: 0.458763 (Recon: 0.458762, Sparse: 0.000001)


...

t-SNE plot saved for Sparse_AutoEncoder
Generating t-SNE plots for Contractive_AutoEncoder...


Extracting embeddings: 100%|██████████| 40/40 [00:02<00:00, 13.66it/s]


t-SNE plot saved for Contractive_AutoEncoder

=== Task 1b: Interpolation Experiment ===
Running interpolation experiment for Sparse_AutoEncoder...
Interpolation experiment completed for Sparse_AutoEncoder
Average PSNR: inf
Average L2 Distance: 1.6567
Running interpolation experiment for Contractive_AutoEncoder...
Interpolation experiment completed for Contractive_AutoEncoder
Average PSNR: inf
Average L2 Distance: 0.0022

=== Task 1c: Classifier Training ===
Training classifiers on embeddings with SVC...


Extracting embeddings: 100%|██████████| 40/40 [00:02<00:00, 18.31it/s]


Training SVM classifiers...
Training Logistic Regression classifiers...

Classifier Results (with SVC and LogReg):
                  Model  SVM Accuracy  Logistic Regression Accuracy
     Sparse AutoEncoder      0.194333                      0.197000
Contractive AutoEncoder      0.113333                      0.113333

=== Task 2: VAE Latent Traversals ===
Generating VAE latent traversals...
Generating random samples...
VAE latent traversals completed

All evaluation tasks completed!
Assignment completed successfully!
